# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -U mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
from pprint import pprint

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Access dataset metadata object
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs. All entities are referenced by their `@id` fields per Croissant specification.

In [ ]:
# List available record sets and their details
print("Available record sets (by @id):")
for rs in dataset.record_sets:
    print(f"@id: {rs['@id']} | name: {rs.get('name', 'N/A')}")

# For demonstration, show the fields for each record set
overview = {}
for rs in dataset.record_sets:
    record_set_id = rs['@id']
    fields = rs.get('field', [])
    # Croissant field can be either a dict or a list
    if isinstance(fields, dict):
        fields = [fields]
    overview[record_set_id] = []
    for field in fields:
        field_id = field['@id'] if isinstance(field, dict) else field
        overview[record_set_id].append(field_id)

# Display overview
for rs_id, fld_list in overview.items():
    print(f"Record set @id: {rs_id}\n  Fields: {fld_list}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# List of record set IDs for extraction
record_sets_ids = [rs['@id'] for rs in dataset.record_sets]
print(f"Record sets to extract: {record_sets_ids}\n")

dataframes = {}
for record_set_id in record_sets_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded DataFrame for record set '{record_set_id}' with shape: {df.shape}")
    except Exception as e:
        print(f"Could not load DataFrame for record set '{record_set_id}': {e}")

# For demonstration, use the first available record set for further exploration
if len(record_sets_ids) > 0:
    record_set_demo = record_sets_ids[0]
    print(f"\nColumns in record set '{record_set_demo}':")
    print(list(dataframes[record_set_demo].columns))
    dataframes[record_set_demo].head()
else:
    print('No record sets found for extraction.')

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes operations like removing outliers, transforming distributions, or grouping by key attributes.

In [ ]:
# For demonstration, attempt to analyze a likely numeric survey field e.g., GAD-7/PHQ-9/PSQ scores
demo_df = dataframes[record_set_demo]
numeric_candidates = [col for col in demo_df.columns if demo_df[col].dtype in ['int64', 'float64']]
print(f"Numeric candidate fields: {numeric_candidates}")

# Pick first candidate (commonly one of the score columns if present, e.g., 'gad7_total_score' or similar)
numeric_field = numeric_candidates[0] if len(numeric_candidates) > 0 else None
if numeric_field:
    print(f"Using numeric field: {numeric_field}")
    # Filter criteria (e.g. scores above a threshold for GAD-7/PHQ-9/PSQ)
    threshold = 10
    filtered_df = demo_df[demo_df[numeric_field] > threshold].copy()
    print(f"Filtered records with {numeric_field} > {threshold}:")
    print(filtered_df.head())

    # Normalize
    mu = filtered_df[numeric_field].mean()
    sigma = filtered_df[numeric_field].std()
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - mu) / sigma
    print(f"Normalized {numeric_field} for filtered records:")
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Group by a likely demographic categorical column (e.g., 'gender', 'age_group', etc.)
    # Try 'gender' or the first string/object column not numeric
    group_candidates = [col for col in demo_df.columns if demo_df[col].dtype == 'object' and col != numeric_field]
    group_field = None
    for candidate in ['gender', 'sex', 'age_group', 'marital_status', 'level_of_education']:
        if candidate in demo_df.columns:
            group_field = candidate
            break
    if group_field is None and len(group_candidates) > 0:
        group_field = group_candidates[0]
    if group_field:
        print(f"Grouping by: {group_field}")
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        print(f"Grouped data by {group_field} (mean {numeric_field}):")
        print(grouped_df.head())
else:
    print("No numeric fields available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset using `matplotlib` or `seaborn`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field:
    plt.figure(figsize=(7,4))
    sns.histplot(demo_df[numeric_field].dropna(), bins=12, kde=True)
    plt.title(f'Distribution of {numeric_field}')
    plt.xlabel(numeric_field)
    plt.ylabel('Frequency')
    plt.show()

    if group_field:
        plt.figure(figsize=(8,4))
        sns.boxplot(x=group_field, y=numeric_field, data=demo_df)
        plt.title(f'{numeric_field} by {group_field}')
        plt.xticks(rotation=45)
        plt.show()
else:
    print("No numeric field for visualization.")

## 6. Conclusion
In this notebook, we loaded and explored the FAIR² Mental Health Survey dataset from Kilifi County, Kenya using the Croissant metadata standard and the `mlcroissant` library. We inspected its record sets and fields by their `@id`, extracted tabular data to pandas, performed basic EDA including filtering and normalization, and produced example visualizations. This workflow demonstrates transparent, reproducible, AI-ready processing for FAIR research collections.

_For more information, visit the [dataset homepage](https://sen.science/doi/10.71728/senscience.vcs2-05nj)._